# SDPO + LoRA 训练示例（BF16）

- 只训练 LoRA 适配器，避免全参显存压力。
- BF16 autocast，无 GradScaler，适配 A100。
- 基于 `eval/core_sdpo.py` 和 `ARChitects/architect_sdpo.py`。

In [1]:
!nvidia-smi

Mon Mar  2 23:54:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          On  |   00000000:3D:00.0 Off |                  Off |
| N/A   34C    P0             63W /  250W |       1MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import os
print(os.getcwd())
os.chdir("/data/coding/ARC")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from torch import amp
from peft import LoraConfig, get_peft_model
from peft.utils import get_peft_model_state_dict

from ARChitects.architect_sdpo import SDPOConfig, load_model_and_tokenizer, load_sdpo_dataset, build_dataloader, SDPOTrainer
from eval.core_sdpo import sdpo_forward, _log_print

/data/coding


/data/miniconda/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# SDPO 基础配置（按需修改路径）
config = SDPOConfig(
    model_path='outputs/arc_lora_sft_C/lora_merged_model',
    tokenizer_path='outputs/arc_lora_sft_C/lora_merged_model',
    report_path='outputs/architect_eval/report_archi_training_training.json',
    dataset_root='data/training',
    batch_size=2,
    learning_rate=1e-4,
    top_k=64,
    max_steps=20000,
)
device = torch.device(config.device)
config

SDPOConfig(model_path='outputs/arc_lora_sft_C/lora_merged_model', tokenizer_path='outputs/arc_lora_sft_C/lora_merged_model', report_path='outputs/architect_eval/report_archi_training_training.json', dataset_root='data/training', batch_size=2, learning_rate=5e-05, weight_decay=0.01, grad_clip=1.0, warmup_steps=10, max_steps=1000, loss_type='js', top_k=64, use_8bit_optimizer=True, device='cuda')

In [4]:
# 加载模型并封装 LoRA
model, tokenizer = load_model_and_tokenizer(config)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=64,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'up_proj', 'down_proj', 'gate_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
model.to(device)

model.gradient_checkpointing_enable()

dataset = load_sdpo_dataset(config)
print(f"trainable examples: {len(dataset)}")
dataloader = build_dataloader(tokenizer, dataset, device=device, batch_size=config.batch_size)
trainer = SDPOTrainer(model, tokenizer, config)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 398/398 [00:00<00:00, 480.94it/s, Materializing param=m


[sdpo] model/tokenizer loaded from outputs/arc_lora_sft_C/lora_merged_model
trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157
[sdpo] load report: outputs/architect_eval/report_archi_training_training.json
[sdpo] scanning record #20 task=0962bcdd
[sdpo] scanning record #40 task=1190bc91
[sdpo] scanning record #60 task=17829a00
[sdpo] scanning record #80 task=1b60fb0c
[sdpo] scanning record #100 task=1f642eb9
[sdpo] scanning record #120 task=22a4bbc2
[sdpo] scanning record #140 task=2697da3f
[sdpo] scanning record #160 task=2c0b0aff
[sdpo] scanning record #180 task=320afe60
[sdpo] scanning record #200 task=363442ee
[sdpo] scanning record #220 task=3c9b0459
[sdpo] scanning record #240 task=42918530
[sdpo] scanning record #260 task=469497ad
[sdpo] scanning record #280 task=4c4377d9
[sdpo] scanning record #300 task=5168d44c
[sdpo] scanning record #320 task=5587a8d0
[sdpo] scanning record #340 task=5adee1b2
[sdpo] scanning record #360 task=6165ea8f
[sdpo] scanni

In [5]:
# BF16 训练循环（LoRA 版本）
def train_loop_bf16(
    trainer: SDPOTrainer,
    train_loader,
    *,
    valid_loader=None,
    max_steps=None,
    grad_accum_steps=2,
    log_every=100,
    eval_every=100,
    ema_beta=0.9,
):
    max_steps = max_steps or trainer.config.max_steps
    step = 0
    train_iter = iter(train_loader)
    valid_iter = iter(valid_loader) if valid_loader is not None else None
    opt = trainer.optimizer
    sch = trainer.lr_scheduler
    opt.zero_grad(set_to_none=True)

    recent_losses = []
    ema_loss = None

    while step < max_steps:
        micro_loss_sum = 0.0
        for _ in range(grad_accum_steps):
            try:
                batch = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                batch = next(train_iter)

            batch = batch.to(trainer.device)
            with amp.autocast("cuda", dtype=torch.bfloat16):
                loss, _ = sdpo_forward(
                    trainer.model,
                    trainer.tokenizer,
                    batch,
                    loss_type=trainer.config.loss_type,
                    top_k=trainer.config.top_k,
                    return_aux=False,
                    keep_only_target_logits=True,
                )
                loss_for_backward = loss / grad_accum_steps

            loss_for_backward.backward()
            micro_loss_sum += loss.item()

        if trainer.config.grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(trainer.model.parameters(), trainer.config.grad_clip)
        opt.step()
        sch.step()
        opt.zero_grad(set_to_none=True)
        step += 1

        step_loss = micro_loss_sum / grad_accum_steps
        recent_losses.append(step_loss)
        if len(recent_losses) > log_every:
            recent_losses.pop(0)
        ema_loss = step_loss if ema_loss is None else (ema_beta * ema_loss + (1 - ema_beta) * step_loss)

        if step % log_every == 0:
            avg_loss = sum(recent_losses) / len(recent_losses)
            _log_print(f"[sdpo] step {step}/{max_steps} loss={step_loss:.4f} avg{len(recent_losses)}={avg_loss:.4f} ema={ema_loss:.4f}")

        if valid_iter is not None and step % eval_every == 0:
            try:
                vbatch = next(valid_iter)
            except StopIteration:
                valid_iter = iter(valid_loader)
                vbatch = next(valid_iter)
            vbatch = vbatch.to(trainer.device)
            with torch.no_grad(), amp.autocast("cuda", dtype=torch.bfloat16):
                v_loss, _ = sdpo_forward(
                    trainer.model,
                    trainer.tokenizer,
                    vbatch,
                    loss_type=trainer.config.loss_type,
                    top_k=trainer.config.top_k,
                    return_aux=False,
                    keep_only_target_logits=True,
                )
            _log_print(f"[sdpo] eval step {step}: loss={v_loss.item():.4f}")

    _log_print(f"[sdpo] training done: steps={step}")
    return trainer


In [ ]:
# 运行若干步示例（可调整步数与日志频率）
_ = train_loop_bf16(
    trainer,
    dataloader,
    valid_loader=None,
    max_steps=1000,
    grad_accum_steps=2,
    log_every=20,
    eval_every=100,
)


[sdpo] forward batch: seq=360 loss=0.1059
[sdpo] forward batch: seq=195 loss=0.1400
[sdpo] forward batch: seq=109 loss=0.0841
[sdpo] forward batch: seq=359 loss=0.1345
[sdpo] forward batch: seq=419 loss=0.0407
[sdpo] forward batch: seq=431 loss=1.0434
[sdpo] forward batch: seq=89 loss=1.6930
[sdpo] forward batch: seq=109 loss=0.1315
[sdpo] forward batch: seq=149 loss=0.4484
[sdpo] forward batch: seq=239 loss=0.1335
[sdpo] forward batch: seq=271 loss=0.0876
[sdpo] forward batch: seq=155 loss=0.1287
[sdpo] forward batch: seq=271 loss=0.3497
[sdpo] forward batch: seq=233 loss=0.2756
[sdpo] forward batch: seq=109 loss=0.0585
[sdpo] forward batch: seq=103 loss=0.4411
[sdpo] forward batch: seq=208 loss=0.0550
[sdpo] drop long samples: dropped=1 kept=1 max_seq_len=3000
[sdpo] forward batch: seq=109 loss=0.0955
[sdpo] forward batch: seq=35 loss=0.5017
[sdpo] forward batch: seq=131 loss=0.1193
[sdpo] forward batch: seq=71 loss=0.0385
[sdpo] forward batch: seq=109 loss=0.1940
[sdpo] forward batc

In [8]:
# （可选）保存 LoRA 适配器到新目录
save_dir = 'outputs/sdpo_lora_adapter_2'
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"LoRA adapter saved to {save_dir}")


LoRA adapter saved to outputs/sdpo_lora_adapter


## Evaluation

In [9]:
from eval.solvers import RawSolver
from eval.models import HuggingFaceBackendTextGenerator
from eval.core_C import GridCodec, ARCDataset, run_evaluation

def get_truth(task, context):
    tests = task.get('test') or []
    if len(tests) != 1:
        return None
    return tests[0].get('output')

def get_task_id(task, context):
    return task.get('task_id', context.get('index'))

codec = GridCodec()
eval_dataset = ARCDataset(root='data', split='evaluation', max_tasks=0)

# 30x30 网格回复长度估计
max_grid = [[0 for _ in range(30)] for _ in range(30)]
reply_text = codec.grid_to_text(max_grid) + '<|im_end|>'
max_new_tokens = len(trainer.tokenizer.encode(reply_text)) + 1
print('max_new_tokens:', max_new_tokens)

trainer.model.eval()
eval_model = HuggingFaceBackendTextGenerator(
    model=trainer.model,
    tokenizer=trainer.tokenizer,
    max_new_tokens=max_new_tokens,
)
solver = RawSolver(model=eval_model, codec=codec)

reports = run_evaluation(
    dataset=eval_dataset,
    solver=solver,
    get_truth=get_truth,
    get_task_id=get_task_id,
    output_dir='outputs/architect_eval/reports_sdpo_lora',
    model_id=config.model_path,
    model_key='sdpo_lora',
    viz_failures=True,
    max_tasks=0,
)

print(reports['summary'])


The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


max_new_tokens: 931
RawSolver prompt for test #0:
<|im_start|>userĊ86Ċ64<|im_end|><|im_start|>assistantĊ868686Ċ646464Ċ686868Ċ464646Ċ868686Ċ646464<|im_end|>Ċ<|im_start|>userĊ79Ċ43<|im_end|><|im_start|>assistantĊ797979Ċ434343Ċ979797Ċ343434Ċ797979Ċ434343<|im_end|>Ċ<|im_start|>userĊ32Ċ78<|im_end|><|im_start|>assistantĊ

RawSolver output for test #0:
323232Ċ787878Ċ232323Ċ878787Ċ323232Ċ787878<|im_end|>

Extracted snippet for test #0:
323232Ċ787878Ċ232323Ċ878787Ċ323232Ċ787878<|im_end|>

Deserializing grid from text:
323232Ċ787878Ċ232323Ċ878787Ċ323232Ċ787878

raw_outputs: [{'grid': [[3, 2, 3, 2, 3, 2], [7, 8, 7, 8, 7, 8], [2, 3, 2, 3, 2, 3], [8, 7, 8, 7, 8, 7], [3, 2, 3, 2, 3, 2], [7, 8, 7, 8, 7, 8]], 'raw': '323232Ċ787878Ċ232323Ċ878787Ċ323232Ċ787878<|im_end|>'}]
[eval] task=00576224 status=correct
RawSolver prompt for test #0:
<|im_start|>userĊ00000000000000Ċ00000008808800Ċ00000000888000Ċ00000888800000Ċ00008808008800Ċ00000008888000Ċ00000000808000Ċ00000088808880Ċ00000080000080Ċ00100000000000Ċ0

KeyboardInterrupt: 